# Module 7: Evals (Optional) (13 min)

> **Optional module.** You've already built and deployed a complete agent in Modules 1–5. This module adds automated evaluation on top of that same agent.

Run automated evaluations against the customer service agent — an **output eval** (is the response good?) and a **trajectory eval** (did the agent follow the right workflow?).

**Prerequisites:** Modules 1-4 completed

In [1]:
!pip install -q -r requirements.txt

# The evals framework calls asyncio.run() internally, which clashes with the
# event loop Jupyter already runs. nest_asyncio patches the loop so the
# synchronous run_evaluations() API works inside a notebook cell.
import nest_asyncio

nest_asyncio.apply()

---

## Part 1: Output Evaluation — Is the Response Good?

The `OutputEvaluator` uses LLM-as-a-judge to score agent responses against expected outputs.

An eval is only useful if it can tell a **good** answer from a **bad** one. To prove that,
we run the same test cases against two agents:

1. ❌ **A weak agent with no tools** — it has no way to look up real customer data, so it
   guesses. The judge should give it **low scores**.
2. ✅ **The real agent with tools** — it looks up the actual data. The judge should give it
   **high scores**.

Seeing the weak agent fail is the point: it shows the rubric is actually doing its job, not
just stamping everything 1.0.

In [2]:
from strands import Agent
from strands_evals import eval_task, Case, Experiment
from strands_evals.evaluators import OutputEvaluator
from customer_service_tools import lookup_customer, get_order_history, process_refund

SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise. Use the available tools to look up customer
information and process requests.

Important: Always verify the customer first, then check orders if needed."""


# The collapsed table shows "..." for the reason column. This helper prints the
# judge's reasoning for every case in plain text so you can see WHY each scored
# the way it did — that "why" is the most valuable part of an LLM-as-judge eval.
def print_reasons(report):
    for i, score in enumerate(report.scores):
        name = report.cases[i].get("name", f"case-{i}")
        mark = "✅" if report.test_passes[i] else "❌"
        reason = report.reasons[i] if i < len(report.reasons) else "(no reason returned)"
        print(f"\n{mark} {name}  (score {score:.2f})")
        print(f"   {reason}")


# Two agents under test: a weak one (no tools) and the real one (with tools).
@eval_task()
def weak_agent():
    """No tools — it can't look up real data, so it guesses. Expect LOW scores."""
    return Agent(
        system_prompt="You are a customer service agent. Answer as best you can.",
        callback_handler=None,
    )


@eval_task()
def good_agent():
    """Has tools — it looks up the real data. Expect HIGH scores."""
    return Agent(
        tools=[lookup_customer, get_order_history, process_refund],
        system_prompt=SYSTEM_PROMPT,
        callback_handler=None,
    )


# Test cases — the same questions for both agents
output_cases = [
    Case[str, str](
        name="order-status-check",
        input="I'm customer C-1001. Where is my USB-C Hub order?",
        expected_output="The USB-C Hub is shipped with tracking TRK-887766, estimated delivery 2025-05-06.",
    ),
    Case[str, str](
        name="delayed-order-empathy",
        input="I'm customer C-1002. My keyboard order is delayed and I'm frustrated!",
        expected_output="Acknowledge frustration, provide order status for the delayed mechanical keyboard with tracking info.",
    ),
    Case[str, str](
        name="unknown-customer",
        input="I'm customer C-9999. What are my orders?",
        expected_output="Inform the customer that no account was found with that ID and ask them to verify.",
    ),
]

# Define the evaluator rubric
output_evaluator = OutputEvaluator(
    rubric="""
    Evaluate the customer service response against the expected output:
    1. Accuracy — Does it contain the correct order/tracking details? Invented or
       missing details (e.g. a made-up tracking number) must score low.
    2. Tone — Is it professional and empathetic?
    3. Completeness — Does it fully address the customer's concern?

    Score 1.0 only if accuracy is correct AND tone and completeness are met.
    Score 0.5 if partially met (e.g. right tone but wrong/missing facts).
    Score 0.0 if the information is inadequate, invented, or incorrect.
    """,
    include_inputs=True,
)

print("❌ Weak agent (no tools) — expect LOW scores:")
weak_report = Experiment[str, str](cases=output_cases, evaluators=[output_evaluator]).run_evaluations(weak_agent)
# Use .display() (static render) in notebooks — .run_display() opens an
# interactive Rich view that blocks the cell waiting for input.
weak_report.display(include_actual_output=True)
print_reasons(weak_report)  # the judge's "why" for each case

print("\n✅ Real agent (with tools) — expect HIGH scores:")
good_report = Experiment[str, str](cases=output_cases, evaluators=[output_evaluator]).run_evaluations(good_agent)
good_report.display(include_actual_output=True)
print_reasons(good_report)

print(
    f"\n📊 No-tools agent: {weak_report.overall_score:.2f}"
    f"  vs  with-tools agent: {good_report.overall_score:.2f}"
)
print("The gap is the eval doing its job — it catches the agent that guesses.")

❌ Weak agent (no tools) — expect LOW scores:


Task was destroyed but it is pending!
task: <Task cancelling name='Task-8' coro=<_poll_cancel_signal() running at /home/john/Downloads/project/centre/.venv/lib/python3.14/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task cancelling name='Task-150' coro=<_poll_cancel_signal() running at /home/john/Downloads/project/centre/.venv/lib/python3.14/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task cancelling name='Task-355' coro=<_poll_cancel_signal() running at /home/john/Downloads/project/centre/.venv/lib/python3.14/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.33           Pass Rate: 0.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                           Test Case Results                                            
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ index ┃ name                  ┃ evaluator       ┃ score ┃ test_pass ┃ reason ┃ input ┃ actual_output ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ▶ 0   │ order-status-check    │ OutputEvaluator │ 0.30  │ ❌        │ ...    │ ...   │ ...           │
├───────┼───────────────────────┼─────────────────┼───────┼───────────┼────────┼───────┼───────────────┤
│ ▶ 1   │ delayed-order-empathy │ OutputEvaluator │ 0.40  │ ❌        │ ...    │ ...   │ ...           │
├───────┼───────────────────────┼─────────────────┼───────┼───────────┼────────┼───────┼───────────────┤
│ ▶ 2   │ unknown-customer      │ OutputEvaluator │ 0.30  │ ❌        │ ...    │ ...   │ ...           │
└───────┴───────────────────────┴─────────────────┴───────┴───────────┴────────┴───────┴───────────────┘


❌ order-status-check  (score 0.30)
   The response fails on accuracy — it does not provide the correct tracking number (TRK-887766) or estimated delivery date (2025-05-06) that were expected. The agent claims it has no access to order information and deflects the customer with generic suggestions, which means the customer's core concern is left unresolved. However, the tone is professional and empathetic, and the response does attempt to guide the customer toward resolution. Because accuracy (the most critical criterion) is entirely missing but tone is good, the score lands below 0.5.

❌ delayed-order-empathy  (score 0.40)
   **Tone (Met):** The response is professional, empathetic, and appropriately acknowledges the customer's frustration. It apologizes sincerely and validates the customer's feelings.

**Accuracy (Not Met):** The expected output requires providing the actual order status and tracking information for customer C-1002's delayed mechanical keyboard. The agent admits it h

Task was destroyed but it is pending!
task: <Task cancelling name='Task-571' coro=<_poll_cancel_signal() running at /home/john/Downloads/project/centre/.venv/lib/python3.14/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task cancelling name='Task-793' coro=<_poll_cancel_signal() running at /home/john/Downloads/project/centre/.venv/lib/python3.14/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task cancelling name='Task-1091' coro=<_poll_cancel_signal() running at /home/john/Downloads/project/centre/.venv/lib/python3.14/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.98           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                           Test Case Results                                            
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ index ┃ name                  ┃ evaluator       ┃ score ┃ test_pass ┃ reason ┃ input ┃ actual_output ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ▶ 0   │ order-status-check    │ OutputEvaluator │ 0.95  │ ✅        │ ...    │ ...   │ ...           │
├───────┼───────────────────────┼─────────────────┼───────┼───────────┼────────┼───────┼───────────────┤
│ ▶ 1   │ delayed-order-empathy │ OutputEvaluator │ 1.00  │ ✅        │ ...    │ ...   │ ...           │
├───────┼───────────────────────┼─────────────────┼───────┼───────────┼────────┼───────┼───────────────┤
│ ▶ 2   │ unknown-customer      │ OutputEvaluator │ 1.00  │ ✅        │ ...    │ ...   │ ...           │
└───────┴───────────────────────┴─────────────────┴───────┴───────────┴────────┴───────┴───────────────┘


✅ order-status-check  (score 0.95)
   All critical facts match the expected output: item (USB-C Hub), status (Shipped), tracking number (TRK-887766), and estimated delivery (May 6, 2025). Additional details like Order ID, price, and order date are consistent with a real system lookup and do not contradict expected data. The name "Sarah" is an extra detail not in the expected output but is a reasonable return from a customer profile lookup. Tone is professional, warm, and empathetic. The response is complete and fully addresses the customer's concern. Minor deduction for the unverifiable "Sarah" name detail and extra fields not confirmed by the expected output, but overall accuracy, tone, and completeness are solidly met.

✅ delayed-order-empathy  (score 1.00)
   The response fully meets all three criteria. (1) Accuracy: Specific, internally consistent order details are provided — customer name (Mike), order ORD-5390, Mechanical Keyboard at $149.99, order/delivery dates, delayed status

---

## Part 2: Trajectory Evaluation — Did It Follow the Workflow?

The `TrajectoryEvaluator` checks that the agent called tools in the **correct order**. For a
refund, the policy is: look up the customer → check their order history → only then process
the refund.

Again we compare two agents to prove the eval works:

1. ❌ **A naive agent** told to "process refunds immediately" — it skips the verification
   steps. The trajectory eval should **fail** it.
2. ✅ **A steered agent** with an explicit ordered workflow — it follows the policy. The
   trajectory eval should **pass** it.

This is how you catch a missing guardrail before it reaches production.

In [3]:
from strands_evals.evaluators import TrajectoryEvaluator
from strands_evals.extractors import tools_use_extractor
from strands_evals.types import TaskOutput

# Two system prompts: one that skips verification, one that enforces the workflow.
NAIVE_PROMPT = """You are a fast customer service agent. When a customer asks for a refund,
process it right away with process_refund. Don't waste time on extra lookups."""

STEERED_PROMPT = """You are a customer service agent. When processing refunds, you MUST:
1. First look up the customer
2. Then check their order history
3. Only then process the refund
Always follow this exact order."""


def run_with(prompt):
    """Build a task function that runs an agent with the given system prompt
    and captures the tools it called."""

    def task(case: Case) -> TaskOutput:
        agent = Agent(
            tools=[lookup_customer, get_order_history, process_refund],
            system_prompt=prompt,
            callback_handler=None,
        )
        response = agent(case.input)
        trajectory = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
        return TaskOutput(output=str(response), trajectory=trajectory)

    return task


# Cases with expected tool sequences
trajectory_cases = [
    Case[str, str](
        name="refund-workflow",
        input="Customer C-1001 wants a refund for order ORD-5521 ($79.99). Process it now.",
        expected_trajectory=["lookup_customer", "get_order_history", "process_refund"],
    ),
    Case[str, str](
        name="info-lookup-only",
        input="Look up customer C-1001 and tell me their order history.",
        expected_trajectory=["lookup_customer", "get_order_history"],
    ),
    Case[str, str](
        name="customer-lookup-only",
        input="Look up customer C-1002's account info.",
        expected_trajectory=["lookup_customer"],
    ),
]

# Create trajectory evaluator
trajectory_evaluator = TrajectoryEvaluator(
    rubric="""
    Evaluate whether the agent followed the expected tool sequence:
    - The expected tools should appear in order (extra tools in between are OK).
    - Score 1.0 if the expected sequence is followed correctly.
    - Score 0.5 if tools are called but in wrong order.
    - Score 0.0 if expected tools are missing entirely.
    """,
    include_inputs=True,
)

# Give evaluator context about available tools
sample_agent = Agent(tools=[lookup_customer, get_order_history, process_refund])
tool_descriptions = tools_use_extractor.extract_tools_description(sample_agent, is_short=True)
trajectory_evaluator.update_trajectory_description(tool_descriptions)

print("❌ Naive agent (skips verification) — expect a FAILED refund trajectory:")
naive_report = Experiment[str, str](cases=trajectory_cases, evaluators=[trajectory_evaluator]).run_evaluations(run_with(NAIVE_PROMPT))
# .display() renders statically; .run_display() would block the cell (see Part 1).
naive_report.display(include_actual_trajectory=True, include_expected_trajectory=True)
print_reasons(naive_report)  # see WHY the refund case failed (defined in Part 1)

print("\n✅ Steered agent (enforced workflow) — expect all trajectories to PASS:")
steered_report = Experiment[str, str](cases=trajectory_cases, evaluators=[trajectory_evaluator]).run_evaluations(run_with(STEERED_PROMPT))
steered_report.display(include_actual_trajectory=True, include_expected_trajectory=True)
print_reasons(steered_report)

print(
    f"\n📊 Naive agent: {naive_report.overall_score:.2f}"
    f"  vs  steered agent: {steered_report.overall_score:.2f}"
)
print("The steering handlers are what close that gap — and the eval is what proves it.")

❌ Naive agent (skips verification) — expect a FAILED refund trajectory:


Task was destroyed but it is pending!
task: <Task cancelling name='Task-1240' coro=<_poll_cancel_signal() running at /home/john/Downloads/project/centre/.venv/lib/python3.14/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.67           Pass Rate: 0.6666666666666666                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 Test Case Results                                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ index ┃ name           ┃ evaluator      ┃ score ┃ test_pass ┃ reason ┃ input ┃ actual_traject… ┃ expected_traj… ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ ▶ 0   │ refund-workfl… │ TrajectoryEva… │ 0.00  │ ❌        │ ...    │ ...   │ ...             │ ...            │
├───────┼────────────────┼────────────────┼───────┼───────────┼────────┼───────┼─────────────────┼────────────────┤
│ ▶ 1   │ info-lookup-o… │ TrajectoryEva… │ 1.00  │ ✅        │ ...    │ ...   │ ...             │ ...            │
├───────┼────────────────┼────────────────┼───────┼───────────┼────────┼───────┼─────────────────┼────────────────┤
│ ▶ 2   │ customer-look… │ TrajectoryEva… │ 1.00  │ ✅        │ ...    │ ...   │ ...             │ ...            │
└───────┴────────────────┴────────────────┴───────┴───────────┴────────┴───────┴─────────────────┴────────────────┘

Task was destroyed but it is pending!
task: <Task cancelling name='Task-1519' coro=<_poll_cancel_signal() running at /home/john/Downloads/project/centre/.venv/lib/python3.14/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task cancelling name='Task-1744' coro=<_poll_cancel_signal() running at /home/john/Downloads/project/centre/.venv/lib/python3.14/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>



❌ refund-workflow  (score 0.00)
   The agent skipped two critical tools in the expected sequence: `lookup_customer` and `get_order_history` were never called. Only `process_refund` was executed, which means the agent jumped straight to processing without first verifying the customer's identity or validating the order history. The in_order_match_scorer returned 0.0 because the expected sequence (lookup_customer → get_order_history → process_refund) was not followed — 2 of 3 required tools were missing entirely. Per the rubric, this warrants a score of 0.0.

✅ info-lookup-only  (score 1.00)
   The agent followed the expected tool sequence exactly. It first called `lookup_customer` with the correct customer ID (C-1001), then called `get_order_history` with the same ID — matching the expected trajectory `['lookup_customer', 'get_order_history']` in the correct order. Both tools returned valid results, and the final output accurately summarizes the data from both tool calls.

✅ customer-lo

Task was destroyed but it is pending!
task: <Task cancelling name='Task-1920' coro=<_poll_cancel_signal() running at /home/john/Downloads/project/centre/.venv/lib/python3.14/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>
Task was destroyed but it is pending!
task: <Task cancelling name='Task-2121' coro=<_poll_cancel_signal() running at /home/john/Downloads/project/centre/.venv/lib/python3.14/site-packages/strands/models/bedrock.py:96> wait_for=<Future cancelled>>


╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 Test Case Results                                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ index ┃ name           ┃ evaluator      ┃ score ┃ test_pass ┃ reason ┃ input ┃ actual_traject… ┃ expected_traj… ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ ▶ 0   │ refund-workfl… │ TrajectoryEva… │ 1.00  │ ✅        │ ...    │ ...   │ ...             │ ...            │
├───────┼────────────────┼────────────────┼───────┼───────────┼────────┼───────┼─────────────────┼────────────────┤
│ ▶ 1   │ info-lookup-o… │ TrajectoryEva… │ 1.00  │ ✅        │ ...    │ ...   │ ...             │ ...            │
├───────┼────────────────┼────────────────┼───────┼───────────┼────────┼───────┼─────────────────┼────────────────┤
│ ▶ 2   │ customer-look… │ TrajectoryEva… │ 1.00  │ ✅        │ ...    │ ...   │ ...             │ ...            │
└───────┴────────────────┴────────────────┴───────┴───────────┴────────┴───────┴─────────────────┴────────────────┘


✅ refund-workflow  (score 1.00)
   The actual trajectory follows the expected tool sequence exactly and in the correct order: (1) lookup_customer → (2) get_order_history → (3) process_refund. All three expected tools are present with no missing steps and no out-of-order calls. The output also correctly reflects the results from each tool call.

✅ info-lookup-only  (score 1.00)
   The agent correctly followed the expected tool sequence. First, `lookup_customer` was called with the customer ID `C-1001` to retrieve the customer's details. Then, `get_order_history` was called with the same customer ID to retrieve the order history. Both tools appear in the exact expected order with no missing tools. The output also accurately reflects the data returned by both tools.

✅ customer-lookup-only  (score 1.00)
   The agent correctly called the `lookup_customer` tool with the appropriate `customer_id` parameter ('C-1002'), exactly matching the expected trajectory. The tool was used in the correc

---

## 🎯 Try It Yourself

You've seen the eval catch a weak agent and a missing guardrail. Now extend it:

- Add a case where **no customer ID is given** — the agent should ask, not guess, so the
  expected trajectory is empty `[]`.
- Or tweak the `NAIVE_PROMPT` above and watch the scores move. That feedback loop — change
  the agent, rerun the eval, compare the numbers — is exactly how you harden an agent.

In [ ]:
# Challenge: Add a case where no customer ID is given
# Expected trajectory should be empty [] — the agent should ask, not guess

# new_case = Case[str, str](
#     name="missing-customer-id",
#     input="I want to return something I bought last week.",
#     expected_trajectory=[],
# )

# Your code here...

---

## What's Next

You've validated the agent works correctly — and you already deployed it back in **Module 5: Deploy**. That completes the workshop. To take the agent to production operations (managed Gateway tools, memory, and observability), see the follow-on workshop.